# Rain and cloud timeline

Inspect hourly precipitation together with total cloud fraction and summarize wet forecast periods.

This live demonstration uses public meteo@uniparthenope data. Values are generated
automatically and are intended for research, education, and general information—not for official
warnings, navigation, or safety-critical decisions. Times are UTC. Run and edit the cells in this
browser-based workspace; your changes stay in your browser unless you download the notebook.


In [ ]:
import json
import os
import sys
from urllib.parse import urlencode
from datetime import datetime

if sys.platform == "emscripten":
    import piplite
    await piplite.install("ipywidgets")

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import HTML, clear_output, display

API_BASE = "https://api.meteo.uniparthenope.it" if sys.platform == "emscripten" else os.environ.get(
    "METEO_API_BASE_URL", "https://api.meteo.uniparthenope.it"
).rstrip("/")
BRAND = "#1f4e79"
ACCENT = "#2a91eb"

def api_json(path, params=None):
    url = f"{API_BASE}/{path.lstrip('/')}"
    if params:
        url = f"{url}?{urlencode(params)}"
    if sys.platform == "emscripten":
        from pyodide.http import open_url
        return json.load(open_url(url))
    import requests
    response = requests.get(url, timeout=20)
    response.raise_for_status()
    return response.json()

def forecast_frame(place):
    payload = api_json(f"products/wrf5/timeseries/{place}")
    if payload.get("result") != "ok":
        raise ValueError(f"No WRF forecast is available for {place}")
    frame = pd.DataFrame(payload["timeseries"])
    frame["time"] = pd.to_datetime(frame["dateTime"], format="%Y%m%dZ%H%M", utc=True)
    return frame.set_index("time")

PLACES = {"Gulf of Naples": "ca001", "Gulf of Salerno": "ca002", "Cilento": "ca003"}


In [ ]:
place = widgets.Dropdown(options=PLACES, description="Area:")
rain_threshold = widgets.FloatSlider(value=1.0, min=0, max=10, step=.5, description="Wet ≥ mm:")
output = widgets.Output()

def render(*_):
    with output:
        clear_output(wait=True)
        try:
            frame = forecast_frame(place.value)
            fig, ax = plt.subplots(figsize=(10, 4.8))
            ax.bar(frame.index, frame.crh, width=.03, color=ACCENT, label="Hourly rain")
            ax.set_ylabel("Rain (mm/hour)")
            cloud = ax.twinx()
            cloud.plot(frame.index, frame.clf * 100, color="#607486", lw=1.5, label="Cloud")
            cloud.set_ylabel("Cloud fraction (%)")
            ax.axhline(rain_threshold.value, color="#d95f02", ls="--", lw=1)
            wet = frame[frame.crh >= rain_threshold.value]
            ax.set_title(f"{place.label}: {len(wet)} forecast hours at or above threshold")
            ax.grid(axis="y", alpha=.25)
            fig.autofmt_xdate()
            plt.show()
            display(wet[["crh", "clf", "t2c", "ws10"]].head(20).rename(columns={"crh": "rain_mm", "clf": "cloud_fraction"}))
        except Exception as exc:
            display(HTML(f"<b>Forecast request failed:</b> {exc}"))

place.observe(render, names="value")
rain_threshold.observe(render, names="value")
display(widgets.VBox([widgets.HBox([place, rain_threshold]), output]))
render()
